# Solving Transverse-field Ising model exactly

Using Numeric methods trying to solve the system exactly as a baseline to compare with NQS and VQE

In [6]:
# Imports 

import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from functools import reduce

## Creating the Hamiltonian for the Transverse Field Ising Model

To start with we need to create the hamiltonian to be able to eventually solve the equation. The hamiltonian can be described by:

$$
\hat{H} = -J \sum_{i=1}^{N-1}\hat{\sigma}^z_i\hat{\sigma}^z_{i+1} - h\sum_{i=1}^N\hat{\sigma}_i^x
$$

We represent this using a sparse matrix and add the option to choose the size N, J, h and the option of periodic and open boundary conditions

In [ ]:

N = 4

s1 = np.zeros(2**N)


def build_hamiltonian(N, J, h, bc = 'Periodic'):
    if bc != 'Periodic' and bc != 'Open':
        raise TypeError

    sx = sp.csr_matrix([
        [0, 1], 
        [1, 0]
        ])
    
    sy = sp.csr_matrix([
        [0,-1j],
        [1j, 0]
        ])
    
    sz = sp.csr_matrix([
        [1, 0],
        [0, -1]
        ])
    
    id2 = sp.eye(2, format='csr')

    
    
    dim = 2**N


    H = sp.csr_matrix((dim, dim), dtype=np.float64)

    for i in range(N):
        ids = [id2] * N 

        ids[i] = sx
        
        term = ids[0]
        for op in ids[1:]:
            term = sp.kron(term, op, format='csr')


        H -= h*term

    num_bonds = N if bc == 'Periodic' else N-1
    for i in range(num_bonds):
        ids = [id2] * N 

        ids[i] = sz 
        
        ids[(i+1) % N] = sz   

        term = ids[0]
        for op in ids[1:]:
            term = sp.kron(term, op, format='csr')




    return H




In [13]:
H = build_hamiltonian(4,1.0,1.0,bc='Periodic')

energies, states = spla.eigsh(H, k=1, which='SA')

E0 = energies[0]
psi= states[:, 0]
print(r'Ground state energy $E_0=$', E0)


Ground state energy $E_0=$ -4.0
